In [1]:
import sys
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Permite importar a pasta src/ estando dentro de notebooks/
sys.path.append(os.path.abspath('..'))

from src.models import MLP
from src.train import treinar
from src.metrics import avaliar_classificacao_multiclasse

def fixar_sementes(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fixar_sementes(42)
print("PyTorch versão:", torch.__version__)

PyTorch versão: 2.15.0+cpu


In [2]:
target_col = 'Addiction_Level'

df_train_full = pd.read_csv('../data/processed/train_classification.csv')
df_test = pd.read_csv('../data/processed/test_classification.csv')

X_full = df_train_full.drop(columns=[target_col]).values
y_full = df_train_full[target_col].values

X_test = df_test.drop(columns=[target_col]).values
y_test = df_test[target_col].values
features_names = df_train_full.drop(columns=[target_col]).columns.tolist()

# 1. Separar validação (20% do treino) de forma estratificada
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# 2. Padronização das features (ajustada APENAS no X_train)
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_val_sc = scaler_X.transform(X_val)
X_test_sc = scaler_X.transform(X_test)

# 3. Converter para Tensores (X em float32, y em long 1D para CrossEntropyLoss)
def to_tensor_clf(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X_tr_t, y_tr_t = to_tensor_clf(X_train_sc, y_train)
X_va_t, y_va_t = to_tensor_clf(X_val_sc, y_val)
X_te_t, y_te_t = to_tensor_clf(X_test_sc, y_test)

# 4. DataLoaders
batch_size = 32
dl_train = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
dl_val = DataLoader(TensorDataset(X_va_t, y_va_t), batch_size=batch_size, shuffle=False)
dl_test = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=batch_size, shuffle=False)

print(f"Treino: {X_train.shape[0]} | Validação: {X_val.shape[0]} | Teste: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]} | Classes: {len(np.unique(y_train))}")

Treino: 960 | Validação: 240 | Teste: 300
Features: 29 | Classes: 4


In [3]:
# Hiperparâmetros Baseline (4 neurônios de saída para as 4 classes)
config_baseline = {
    "in_dim": X_train.shape[1],
    "out_dim": 4,
    "hidden": [32, 32],
    "activation": "ReLU",
    "dropout": 0.0
}

lr = 1e-3
epocas = 200
paciencia = 20
loss_fn = nn.CrossEntropyLoss()

sementes = [42, 100, 2024, 7, 999]
historicos = []
modelos_treinados = []

for semente in sementes:
    fixar_sementes(semente)
    modelo = MLP(**config_baseline)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    
    print(f"Treinando semente {semente}...")
    hist = treinar(modelo, dl_train, dl_val, loss_fn, otimizador, n_epocas=epocas, paciencia=paciencia)
    
    historicos.append(hist)
    modelos_treinados.append(modelo)
    
    melhor_epoca = len(hist["val"]) - paciencia
    melhor_val = min(hist["val"])
    print(f" -> Parou na época {len(hist['val'])} (Melhor Val Loss: {melhor_val:.4f} na época {melhor_epoca})")

# Seleciona o modelo com menor perda de validação entre as sementes
melhor_idx = np.argmin([min(h["val"]) for h in historicos])
modelo_vencedor = modelos_treinados[melhor_idx]
print(f"\nModelo escolhido para salvar: Semente {sementes[melhor_idx]}")

Treinando semente 42...
 -> Parou na época 34 (Melhor Val Loss: 0.5993 na época 14)
Treinando semente 100...
 -> Parou na época 34 (Melhor Val Loss: 0.5599 na época 14)
Treinando semente 2024...
 -> Parou na época 37 (Melhor Val Loss: 0.5797 na época 17)
Treinando semente 7...
 -> Parou na época 34 (Melhor Val Loss: 0.5607 na época 14)
Treinando semente 999...
 -> Parou na época 32 (Melhor Val Loss: 0.5994 na época 12)

Modelo escolhido para salvar: Semente 100


In [4]:
os.makedirs('../models', exist_ok=True)
caminho_modelo = '../models/modelo_classificacao.pt'

checkpoint = {
    "task": "classification",
    "state_dict": modelo_vencedor.state_dict(),
    "config": config_baseline,
    "features": features_names,
    "x_media": torch.tensor(scaler_X.mean_, dtype=torch.float32),
    "x_desvio": torch.tensor(scaler_X.scale_, dtype=torch.float32)
}

torch.save(checkpoint, caminho_modelo)
print(f"Checkpoint completo salvo em: {caminho_modelo}")

Checkpoint completo salvo em: ../models/modelo_classificacao.pt


In [5]:
modelo_vencedor.eval()

with torch.inference_mode():
    logits = modelo_vencedor(X_te_t)
    probs = torch.softmax(logits, dim=1).numpy()

metricas = avaliar_classificacao_multiclasse(y_test, probs, classes=[0, 1, 2, 3])

print("Métricas no Conjunto de Teste (Classificação Multiclasse):")
for k, v in metricas.items():
    if k == "Conf_Matrix":
        print("\nMatriz de Confusão:")
        print(np.array(v))
    else:
        print(f"  {k}: {v:.4f}")

Métricas no Conjunto de Teste (Classificação Multiclasse):
  Accuracy: 0.7333
  Precision_macro: 0.7349
  Recall_macro: 0.7310
  F1_macro: 0.7328
  F1_weighted: 0.7337
  ROC_AUC_macro: 0.9261
  PR_AUC_macro: 0.7748

Matriz de Confusão:
[[46  0 14 12]
 [ 0 70 14  0]
 [13 13 64  0]
 [13  0  1 40]]
